# 📖 Overview

This project is a step-by-step implementation of the original **Transformer Encoder-Decoder** architecture proposed in the landmark paper **"Attention Is All You Need"**.

The objective of this notebook is to understand **how Transformers actually work internally**, instead of treating them as a black box provided by modern deep learning frameworks.

Every major component has been implemented manually using **NumPy**, making this repository ideal for students, ML enthusiasts, and anyone interested in understanding the mathematics and workflow behind Transformer models.

> **Note:** This implementation is designed for educational purposes and focuses on clarity, mathematical intuition, and understanding the architecture rather than production-level optimization.


In [ ]:
import numpy as np

class EncoderBlock:

  def __init__(self, text, embedding_dim=512, seed=42) -> None:
      np.random.seed(seed)    # to keep randomness parmanent

      self.text = text
      self.y_embedding = None
      self.tokens = text.lower().split()

      self.vocab = sorted(set(self.tokens))
      self.vocab = {word: idx for idx, word in enumerate(self.vocab)}

      # Random embedding matrix
      self.embedding_matrix = np.random.randn(len(self.vocab), embedding_dim)

      # Convert sentence to embeddings
      self.embeddings = np.array([
          self.embedding_matrix[self.vocab[word]]
          for word in self.tokens
      ])
      self.dmodel = self.embeddings.shape[1]

      # initialize weights for query, key and value matrix
      self.w_Q = np.random.randn(self.dmodel, self.dmodel)
      self.w_K = np.random.randn(self.dmodel, self.dmodel)
      self.w_V = np.random.randn(self.dmodel, self.dmodel)

      self.encode_position()

      # feed forward network
      self.d_ff = 4 * self.dmodel

      self.w1 = np.random.randn(self.dmodel, self.d_ff)
      self.b1 = np.zeros((1, self.d_ff))

      self.w2 = np.random.randn(self.d_ff, self.dmodel)
      self.b2 = np.zeros((1, self.dmodel))

      # Layer Normalization parameters
      self.gamma = np.ones((1, self.dmodel))
      self.beta = np.zeros((1, self.dmodel))
      self.eps = 1e-5

  # Positional encoding on Embeddings
  def encode_position(self):
    final_encoding = []
    positions = np.arange(len(self.tokens))

    for i in range(self.dmodel // 2):
      omega = 1 / (10000 ** ((2 * i) / self.dmodel))

      # getting encodings on sin / cos functions
      pos_encoding0 = np.sin(omega * positions)
      pos_encoding1 = np.cos(omega * positions)

      # appending encodings
      final_encoding.extend([pos_encoding0, pos_encoding1])

    encoding = np.array(final_encoding)

    # add positional encoding with word's embedding
    self.PE_vector = self.embeddings + encoding.T

  # Activation Function
  def relu(self, x):
    return np.maximum(0, x)

  # function to calculate Softmax
  def softmax(self, x):
      exp = np.exp(x - np.max(x, axis=-1, keepdims=True))
      return exp / np.sum(exp, axis=-1, keepdims=True)

  # Layers Normalization
  def layer_norm(self, x):
    # Mean of each token
    mean = np.mean(x, axis=-1, keepdims=True)

    # Variance of each token
    variance = np.var(x, axis=-1, keepdims=True)

    # Normalize
    x_hat = (x - mean) / np.sqrt(variance + self.eps)

    # Scale and shift
    return self.gamma * x_hat + self.beta

  def feed_forward(self, x):

    hidden = x @ self.w1 + self.b1
    hidden = self.relu(hidden)

    output = hidden @ self.w2 + self.b2
    return output

  # Main Block
  def forward(self):
    # Generate Q, K, V
    Q = self.PE_vector @ self.w_Q
    K = self.PE_vector @ self.w_K
    V = self.PE_vector @ self.w_V

    # Compute Attention Scores
    d_k = K.shape[1]
    scores = (Q @ K.T) / np.sqrt(d_k)

    # Softmax
    weights = self.softmax(scores)

    # Weighted Sum of Values + Normalization
    attention_output = self.PE_vector + weights @ V
    attention_output = self.layer_norm(attention_output)

    # Feed Forward Network
    ffn_output = self.feed_forward(attention_output)

    # Second Residual
    output = attention_output + ffn_output
    output = self.layer_norm(output)

    return output

In [ ]:
class DecoderBlock:

  def __init__(self, text, embedding_dim=512, seed=42) -> None:
      np.random.seed(seed)    # to keep randomness parmanent

      self.text = text
      self.y_embedding = None
      self.tokens = text.lower().split()

      self.vocab = sorted(set(self.tokens))
      self.vocab = {word: idx for idx, word in enumerate(self.vocab)}

      # Random embedding matrix
      self.embedding_matrix = np.random.randn(len(self.vocab), embedding_dim)

      # Convert sentence to embeddings
      self.embeddings = np.array([
          self.embedding_matrix[self.vocab[word]]
          for word in self.tokens
      ])
      self.dmodel = self.embeddings.shape[1]

      # initialize weights for query, key and value matrix
      self.w_Q = np.random.randn(self.dmodel, self.dmodel)
      self.w_K = np.random.randn(self.dmodel, self.dmodel)
      self.w_V = np.random.randn(self.dmodel, self.dmodel)

      self.encode_position()

      # feed forward network
      self.d_ff = 4 * self.dmodel

      self.w1 = np.random.randn(self.dmodel, self.d_ff)
      self.b1 = np.zeros((1, self.d_ff))

      self.w2 = np.random.randn(self.d_ff, self.dmodel)
      self.b2 = np.zeros((1, self.dmodel))

      # Layer Normalization parameters
      self.gamma = np.ones((1, self.dmodel))
      self.beta = np.zeros((1, self.dmodel))
      self.eps = 1e-5

  # Positional encoding on Embeddings
  def encode_position(self):
    final_encoding = []
    positions = np.arange(len(self.tokens))

    for i in range(self.dmodel // 2):
      omega = 1 / (10000 ** ((2 * i) / self.dmodel))

      # getting encodings on sin / cos functions
      pos_encoding0 = np.sin(omega * positions)
      pos_encoding1 = np.cos(omega * positions)

      # appending encodings
      final_encoding.extend([pos_encoding0, pos_encoding1])

    encoding = np.array(final_encoding)

    # add positional encoding with word's embedding
    self.PE_vector = self.embeddings + encoding.T

  # Activation Function
  def relu(self, x):
    return np.maximum(0, x)

  # function to calculate Softmax
  def softmax(self, x):
      exp = np.exp(x - np.max(x, axis=-1, keepdims=True))
      return exp / np.sum(exp, axis=-1, keepdims=True)

  # Layers Normalization
  def layer_norm(self, x):
    # Mean of each token
    mean = np.mean(x, axis=-1, keepdims=True)

    # Variance of each token
    variance = np.var(x, axis=-1, keepdims=True)

    # Normalize
    x_hat = (x - mean) / np.sqrt(variance + self.eps)

    # Scale and shift
    return self.gamma * x_hat + self.beta

  def feed_forward(self, x):

    hidden = x @ self.w1 + self.b1
    hidden = self.relu(hidden)

    output = hidden @ self.w2 + self.b2
    return output

  # Main Block
  def forward(self, encoder_output):
    # Generate Q, K, V
    Q = self.PE_vector @ self.w_Q
    K = self.PE_vector @ self.w_K
    V = self.PE_vector @ self.w_V
    seq_len_tgt = len(self.embeddings)

    # 1.Compute Attention Scores
    d_k = K.shape[1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    # masking
    mask = np.triu(
        np.full((seq_len_tgt, seq_len_tgt), -np.inf),
        k=1
    )
    scores = scores + mask

    # Softmax
    weights = self.softmax(scores)

    # Weighted Sum of Values + Normalization
    attention_output = self.PE_vector + weights @ V
    attention_output = self.layer_norm(attention_output)

    # 2.Cross Attention
    Q = attention_output @ self.w_Q
    K = encoder_output @ self.w_K
    V = encoder_output @ self.w_V

    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = self.softmax(scores)

    # Second Residual Connection
    cross_attention_output = attention_output + weights @ V
    cross_attention_output = self.layer_norm(cross_attention_output)

    # 3.Feed Forward Network
    ffn_output = self.feed_forward(cross_attention_output)

    # Third Residual
    output = cross_attention_output + ffn_output
    output = self.layer_norm(output)

    return output

In [ ]:
encoder_model = EncoderBlock('How are you')
encoder_output = encoder_model.forward()

In [ ]:
decoder_model = DecoderBlock('आप कैसे हैं')
decoder_output = decoder_model.forward(encoder_output)

In [ ]:
decoder_output

array([[-0.3435113 ,  0.25712346,  1.57140372, ..., -0.03684857,
         0.34856715, -2.12403911],
       [-0.34252145,  0.25609105,  1.57267701, ..., -0.03730364,
         0.34815779, -2.12076659],
       [-0.33238516,  0.30222127,  1.63440753, ..., -0.03506047,
         0.30763711, -2.17256895]])